Get data and preprocessing

In [1]:
#import package
import pandas as pd
from src.dataset import loader, data_info
from src.eda import data_profiling
from src.dictionaries import dictionary
from src.pre_processing import fillna_value
from sklearn.impute import KNNImputer

trigger_api = input()
if trigger_api == 1:
    #Get data
    print("Getting data from API")
    article_id = "23690811"
    api_url = f"https://api.figshare.com/v2/articles/{article_id}"
    dataset = loader.get_data(api_url)
    data_info.dataset_info(dataset)
    TravelDataUsers = dataset["B_TravelDataUsers.csv"]
    TravelDataJourneys = dataset["A_TravelDataJourneys.csv"]
else:
    TravelDataJourneys = pd.read_csv(r'D:\MASTER\2.1. ADM\FINAL_PROJECT\CTMJ_ds\A_TravelDataJourneys.csv')
    TravelDataUsers = pd.read_csv(r'D:\MASTER\2.1. ADM\FINAL_PROJECT\CTMJ_ds\B_TravelDataUsers.csv')

print('All dataset are ready for use')

#Preparing data for Clustering
dictionary = dictionary()
touchpoint_mapping = dictionary.type_touch

All dataset are ready for use


In [2]:
#Preparing data for Clustering
user_data = TravelDataUsers[~TravelDataUsers['SPSS_Regio5'].isnull()]
selected_column = [
    'UserID',
    'GenderID', #gioi tinh
    'Age', #tuoi
    'SPSS_Regio5', #vung
    'BAS_huishoudgrootte', #quy mo ho gia dinh (so nguoi trong nha)
    'BAS_werkzaamheid_resp', #nghe nghiep
    'BAS_bruto_jaarinkomen', #tong thu nhap truoc thue
    'afg_kinderen_huishouden', #so luong tre em trong gia dinh
    'AFG_sk2015', #tang lop xa hoi
    'BAS_voltooide_opleiding8_resp', #trinh do hoc van
    'SPSS_Lifestage' #giai doan cuoc doi
]

user_data = user_data[selected_column]

imputer = KNNImputer(n_neighbors=5)
# 3. Fit và Transform
# Kết quả trả về là một Numpy Array (mất tên cột và index)
user_data_imputed_array = imputer.fit_transform(user_data)

# 4. Chuyển về DataFrame
# Sử dụng lại index và columns của dataframe cũ để giữ đúng cấu trúc
user_data = pd.DataFrame(
    user_data_imputed_array,
    columns=user_data.columns,
    index=user_data.index
)

#user_data info
print(user_data.info())
print(user_data.head())

#Preparing data for Prediction
#Dictionary mapping
journey_data = TravelDataJourneys[TravelDataJourneys['Duration'] > 0].copy()
#Duration > 0
journey_data['TIMESPSS'] = pd.to_datetime(journey_data['TIMESPSS'])
journey_data = journey_data.sort_values(by=['UserID', 'TIMESPSS']).reset_index(drop=True)

#Đánh dấu các click liên tiếp cùng loại của cùng 1 User
change_mask = (journey_data['type_touch'] != journey_data['type_touch'].shift()) | (journey_data['UserID'] != journey_data['UserID'].shift())
journey_data['group_id'] = change_mask.cumsum()

#Aggregation
journey_data_compressed = journey_data.groupby(['UserID', 'group_id', 'type_touch']).agg({
    'TIMESPSS': 'first',
    'Duration': 'sum',
    'PurchaseID': 'first',
    'DEVICE_TYPE': 'first',
    'purchase_own': 'max',
    'purchase_any': 'max',
    'MobilePanel': 'first',
    'FixedPanel': 'first'
}).reset_index()

# 5. Mapping tên điểm chạm
journey_data_compressed['touchpoint_name'] = journey_data_compressed['type_touch'].map(touchpoint_mapping)

user_id_list = user_data['UserID'].unique()
journey_data_compressed = journey_data_compressed[journey_data_compressed['UserID'].isin(user_id_list)]

# # 6. Dọn dẹp và kiểm tra
journey_data_compressed = journey_data_compressed.drop(columns='group_id')

print(f"Số dòng sau khi nén: {len(journey_data_compressed)}")

#journey_data info
print(journey_data_compressed.info())
print(journey_data_compressed.head())

<class 'pandas.core.frame.DataFrame'>
Index: 8075 entries, 2 to 9677
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   UserID                         8075 non-null   float64
 1   GenderID                       8075 non-null   float64
 2   Age                            8075 non-null   float64
 3   SPSS_Regio5                    8075 non-null   float64
 4   BAS_huishoudgrootte            8075 non-null   float64
 5   BAS_werkzaamheid_resp          8075 non-null   float64
 6   BAS_bruto_jaarinkomen          8075 non-null   float64
 7   afg_kinderen_huishouden        8075 non-null   float64
 8   AFG_sk2015                     8075 non-null   float64
 9   BAS_voltooide_opleiding8_resp  8075 non-null   float64
 10  SPSS_Lifestage                 8075 non-null   float64
dtypes: float64(11)
memory usage: 757.0 KB
None
   UserID  GenderID   Age  SPSS_Regio5  BAS_huishoudgrootte  \
2     1

In [10]:
# from src.eda import data_profiling
# #Journey Dashboard information
# output_dir = "../../resources/profiling"
# processed_journey_output_filename = "Processed_user_data_profiling.html"
# processed_journey_dashboard_title = "Processed Travel Users Dataset Profiling"
#
# #Run
# data_profiling(user_data, output_dir,processed_journey_output_filename, processed_journey_dashboard_title)

Data Scaling

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# --- 1. PHẦN PREPROCESSING (Giữ nguyên của bạn) ---
robust_features = ['BAS_huishoudgrootte', 'BAS_werkzaamheid_resp', 'afg_kinderen_huishouden', 'BAS_voltooide_opleiding8_resp', 'SPSS_Lifestage']
minmax_features = ['GenderID', 'SPSS_Regio5', 'BAS_bruto_jaarinkomen', 'AFG_sk2015']
standard_features = ['Age']

user_data_preprocessor = ColumnTransformer(
    transformers=[
        ('rob', RobustScaler(), robust_features),
        ('mms', MinMaxScaler(), minmax_features),
        ('std', StandardScaler(), standard_features)
    ],
    remainder='drop'
)

X_scaled = user_data_preprocessor.fit_transform(user_data)



In [5]:
import os
import joblib

# Giả sử bạn đã fit preprocessor này với dữ liệu huấn luyện (X_train)
# user_data_preprocessor.fit(X_train)

# 1. Định nghĩa thư mục lưu trữ (bạn có thể lưu cùng thư mục với model)
model_path = '../../resources/model/clustering/scaler/'
if not os.path.exists(model_path):
    os.makedirs(model_path)

# 2. Lưu Preprocessor bằng joblib
preprocessor_file = os.path.join(model_path, 'user_data_preprocessor.pkl')
joblib.dump(user_data_preprocessor, preprocessor_file)

print("✅ Đã xuất Preprocessor vào thư mục thành công!")

✅ Đã xuất Preprocessor vào thư mục thành công!


In [12]:
import warnings
warnings.filterwarnings('ignore', message='n_jobs value 1 overridden')

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_result = tsne.fit_transform(X_scaled)

# UMAP
# n_neighbors: số lượng láng giềng (thấp = tập trung cục bộ, cao = tập trung toàn cục)
# min_dist: khoảng cách tối thiểu giữa các điểm trong không gian thấp chiều
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
umap_result = reducer.fit_transform(X_scaled)

# --- 3. TẠO DATAFRAME KẾT QUẢ ---
data_viz = pd.DataFrame({
    'PCA1': pca_result[:, 0],
    'PCA2': pca_result[:, 1],
    'TSNE1': tsne_result[:, 0],
    'TSNE2': tsne_result[:, 1],
    'UMAP1': umap_result[:, 0],
    'UMAP2': umap_result[:, 1]
})

# --- 4. TRỰC QUAN HÓA SO SÁNH ---
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# Biểu đồ PCA
sns.scatterplot(ax=axes[0], x='PCA1', y='PCA2', data=data_viz, alpha=0.4, s=10)
axes[0].set_title('PCA - Tổng quan phương sai', fontsize=14)

# Biểu đồ t-SNE
sns.scatterplot(ax=axes[1], x='TSNE1', y='TSNE2', data=data_viz, alpha=0.4, s=10)
axes[1].set_title('t-SNE - Cụm mật độ cục bộ', fontsize=14)

# Biểu đồ UMAP
sns.scatterplot(ax=axes[2], x='UMAP1', y='UMAP2', data=data_viz, alpha=0.4, s=10)
axes[2].set_title('UMAP - Cấu trúc toàn cục & cục bộ', fontsize=14)

plt.suptitle('So sánh các phương pháp trực quan hóa cấu trúc dữ liệu', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

In [13]:
from kneed import KneeLocator
import numpy as np
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

def find_optimal_eps(data, k=20):
    # 1. Tính toán khoảng cách K-neighbors tương tự như trên
    neigh = NearestNeighbors(n_neighbors=k)
    nbrs = neigh.fit(data)
    distances, _ = nbrs.kneighbors(data)

    # Lấy khoảng cách láng giềng xa nhất và sắp xếp
    sorted_distances = np.sort(distances[:, k-1], axis=0)
    x_values = np.arange(len(sorted_distances))

    # 2. Sử dụng KneeLocator để tìm điểm gãy
    # curve='convex' vì đường cong lõm lên trên, direction='increasing'
    kneedle = KneeLocator(x_values, sorted_distances,
                          curve='convex',
                          direction='increasing',
                          interp_method='polynomial')

    eps_opt = sorted_distances[kneedle.knee]

    # 3. Vẽ đồ thị kết quả
    plt.figure(figsize=(10, 6))
    plt.plot(sorted_distances, label='K-distance')
    plt.axhline(y=eps_opt, color='r', linestyle='--', label=f'Optimal Eps: {eps_opt:.4f}')
    plt.axvline(x=kneedle.knee, color='g', linestyle='--', label=f'Knee Point: {kneedle.knee}')

    plt.title(f'Automated Elbow Detection (k={k})')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return eps_opt

# Chạy thử
optimal_eps = find_optimal_eps(X_scaled, k=20)
print(f"Giá trị Eps gợi ý: {optimal_eps}")

In [15]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from collections import Counter

# 1. Cấu hình mô hình theo biểu đồ Elbow của bạn
eps_optimal = 1.1268
min_samples_optimal = 20

dbscan = DBSCAN(eps=eps_optimal, min_samples=min_samples_optimal)
min_samples_optimal = 20

dbscan = DBSCAN(eps=eps_optimal, min_samples=min_samples_optimal)
dbscan_labels = dbscan.fit_predict(X_scaled)
user_data['dbscan_labels'] = dbscan_labels

# 2. Thống kê số lượng
counts = Counter(dbscan_labels)
n_noise = counts.get(-1, 0)
n_clusters = len([k for k in counts.keys() if k != -1])
total_points = len(X_scaled)
noise_percentage = (n_noise / total_points) * 100

print(f"--- KẾT QUẢ DBSCAN (eps={eps_optimal}) ---")
print(f"Số lượng nhiễu (Noise): {n_noise} / {total_points} điểm")
print(f"Tỷ lệ nhiễu: {noise_percentage:.2f}%")
print(f"Số lượng cụm tìm thấy: {n_clusters}")
print(f"Chi tiết các cụm: {dict(sorted(counts.items()))}")

--- KẾT QUẢ DBSCAN (eps=1.1268) ---
Số lượng nhiễu (Noise): 268 / 8075 điểm
Tỷ lệ nhiễu: 3.32%
Số lượng cụm tìm thấy: 3
Chi tiết các cụm: {-1: 268, 0: 4735, 1: 3049, 2: 23}


In [16]:
# Xác định các chỉ số (index) không phải là nhiễu
mask_clean = dbscan_labels != -1

# Tạo tập dữ liệu sạch
X_clean = X_scaled[mask_clean]

# Lưu trữ lại thông tin user tương ứng nếu cần (để mapping lại sau này)
# user_data_clean = user_data.iloc[mask_clean].copy()

print(f"Dữ liệu ban đầu: {len(X_scaled)}")
print(f"Dữ liệu sau khi lọc nhiễu: {len(X_clean)}")

Dữ liệu ban đầu: 8075
Dữ liệu sau khi lọc nhiễu: 7807


In [ ]:
# import os
# import joblib
#
# model_path = '../../resources/model/clustering/'
# if not os.path.exists(model_path):
#     os.makedirs(model_path)
#
# # Lưu Model
# joblib.dump(dbscan, os.path.join(model_path, 'dbscan_clustering_model.pkl'))
#
# print("✅ Đã xuất Model vào thư mục thành công!")

In [ ]:
# import os
# import joblib
#
# # 1. Định nghĩa đường dẫn tới file model đã lưu
# model_path = '../../resources/model/clustering/dbscan_clustering_model.pkl'
#
# # 2. Kiểm tra xem file có tồn tại không trước khi load (để tránh lỗi)
# if os.path.exists(model_path):
#     # Gọi lại model
#     dbscan_loaded = joblib.load(model_path)
#     print("✅ Đã tải (load) Model thành công!")
# else:
#     print(f"❌ Không tìm thấy file model tại đường dẫn: {model_path}")
#
# # 3. Sử dụng model đã load để predict dữ liệu mới
# # Giả sử bạn có dữ liệu mới là 'X_new'
# # Lưu ý: Với DBSCAN trong sklearn, bạn dùng .fit_predict(X_new) hoặc truy cập các thuộc tính cũ.
# # labels = dbscan_loaded.fit_predict(X_new)

In [ ]:
from yellowbrick.cluster import KElbowVisualizer
from sklearn.metrics import davies_bouldin_score
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

def visualizer_all_metrics(X_scaled):
    fig, ax = plt.subplots(1, 3, figsize=(24, 6))
    k_range = range(2, 12)

    # 1. Elbow Method (Distortion Score)
    model1 = KMeans(init='k-means++', random_state=42, n_init=10)
    visualizer1 = KElbowVisualizer(model1, k=(2, 12), ax=ax[0], force_model=True)
    visualizer1.fit(X_scaled)
    k_elbow = visualizer1.elbow_value_
    ax[0].set_title(f"Elbow Method (Optimal k={k_elbow})")

    # 2. Silhouette Score
    model2 = KMeans(init='k-means++', random_state=42, n_init=10)
    visualizer2 = KElbowVisualizer(model2, k=(2, 12), metric='silhouette', ax=ax[1], force_model=True)
    visualizer2.fit(X_scaled)
    k_silhouette = visualizer2.elbow_value_
    ax[1].set_title(f"Silhouette Score (Optimal k={k_silhouette})")

    # 3. Davies-Bouldin Index (DBI)
    dbi_scores = []
    for k in k_range:
        kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        dbi_scores.append(davies_bouldin_score(X_scaled, labels))

    best_dbi_idx = dbi_scores.index(min(dbi_scores))
    k_dbi = k_range[best_dbi_idx]

    # Vẽ biểu đồ DBI
    ax[2].plot(k_range, dbi_scores, marker='o', color='#1313FF', linewidth=2)
    ax[2].set_xlabel('k')
    ax[2].set_ylabel('DBI Score')
    ax[2].set_title(f'Davies-Bouldin Index (Optimal k={k_dbi})')
    ax[2].axvline(k_dbi, color='black', linestyle='--', label=f'Best k = {k_dbi}')
    ax[2].grid(True, alpha=0.3)
    ax[2].legend()

    plt.tight_layout()
    plt.show()

# Gọi hàm với dữ liệu đã chuẩn hóa
visualizer_all_metrics(X_clean)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', message='n_jobs value 1 overridden')

def plot_clustering_results(viz_df, labels, title):
    """
    Vẽ 3 subplot PCA, t-SNE, UMAP tô màu theo nhãn cụm vừa tìm được.
    """
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))

    # Thiết lập màu sắc: -1 (nhiễu) sẽ có màu xám, các cụm còn lại dùng bảng màu Spectral
    unique_labels = set(labels)
    palette = sns.color_palette("Spectral", n_colors=len(unique_labels))
    if -1 in unique_labels:
        palette = {lbl: palette[i] if lbl != -1 else (0.8, 0.8, 0.8) for i, lbl in enumerate(unique_labels)}

    methods = [('PCA1', 'PCA2', 'PCA'), ('TSNE1', 'TSNE2', 't-SNE'), ('UMAP1', 'UMAP2', 'UMAP')]

    for i, (x, y, name) in enumerate(methods):
        sns.scatterplot(
            ax=axes[i], x=x, y=y, data=viz_df,
            hue=labels, palette=palette,
            s=12, alpha=0.6, legend=None
        )
        axes[i].set_title(f"{name} Visualization", fontsize=12)
        axes[i].grid(True, linestyle='--', alpha=0.4)

    plt.suptitle(title, fontsize=15, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()


In [ ]:
import time
import pandas as pd
import numpy as np
import hdbscan
import umap
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import (KMeans, SpectralClustering, AgglomerativeClustering)
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score

def run_clustering_benchmark_with_viz(data, k_list=[2, 3, 4]):

    print("⏳Chuẩn bị không gian trực quan (PCA, t-SNE, UMAP)...")
    pca_res = PCA(n_components=2).fit_transform(data)
    tsne_res = TSNE(n_components=2, random_state=42).fit_transform(data)
    umap_res = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(data)

    viz_df = pd.DataFrame({
        'PCA1': pca_res[:, 0], 'PCA2': pca_res[:, 1],
        'TSNE1': tsne_res[:, 0], 'TSNE2': tsne_res[:, 1],
        'UMAP1': umap_res[:, 0], 'UMAP2': umap_res[:, 1]
    })

    results = []

    def execute_and_log(name, model, params):
        start_time = time.time()
        try:
            labels = model.fit_predict(data)
            end_time = time.time()
            runtime = end_time - start_time

            counts = Counter(labels)
            n_noise = counts.pop(-1, 0)
            n_clusters = len(counts)
            cluster_sizes = {f"C{k}": v for k, v in sorted(counts.items())}

            if n_clusters > 1:
                mask = labels != -1
                s_score = silhouette_score(data[mask], labels[mask])
                dbi_score = davies_bouldin_score(data[mask], labels[mask])

                results.append({
                    "Algorithm": name, "Parameters": params, "n_clusters": n_clusters,
                    "Cluster_Sizes": cluster_sizes, "Noise_Count": n_noise,
                    "Silhouette": round(s_score, 4), "DBI": round(dbi_score, 4),
                    "Runtime (s)": round(runtime, 4)
                })

                title = f"{name} | Params: {params} | Sil: {round(s_score, 3)} | DBI: {round(dbi_score, 4)} | Noise: {n_noise}"
                plot_clustering_results(viz_df, labels, title)
                print(f"Hoàn thành {name} | Clusters: {n_clusters}")

        except Exception as e:
            print(f"Lỗi {name} ({params}): {e}")

    for k in k_list:
        models_k = {
            "K-Means": KMeans(n_clusters=k, random_state=42, n_init=10),
            "GMM": GaussianMixture(n_components=k, random_state=42),
            "Agglomerative": AgglomerativeClustering(n_clusters=k),
            "Spectral_NN_20": SpectralClustering(
            n_clusters=k, affinity='nearest_neighbors', n_neighbors=20, random_state=42),
            "Spectral_NN_50": SpectralClustering(
                n_clusters=k, affinity='nearest_neighbors', n_neighbors=50, random_state=42),
            "Spectral_RBF_high": SpectralClustering(
                n_clusters=k, affinity='rbf', gamma=1.0, random_state=42),
            "Spectral_RBF_mid": SpectralClustering(
                n_clusters=k, affinity='rbf', gamma=0.1, random_state=42),
            "Spectral_RBF_low": SpectralClustering(
                n_clusters=k, affinity='rbf', gamma=0.01, random_state=42),
            "Spectral_Laplacian": SpectralClustering(
                n_clusters=k, affinity='laplacian', random_state=42)
            }
        for name, model in models_k.items():
            execute_and_log(name, model, {"k": k})
    return pd.DataFrame(results)

df_final = run_clustering_benchmark_with_viz(X_clean)

print("\n--- BẢNG SO SÁNH---")
print(df_final.sort_values("Silhouette", ascending=False))
df_final.to_excel('D:/training_result_cleaned.xlsx')

In [ ]:
# df_final_not_cleaned = run_clustering_benchmark_with_viz(X_scaled)
# df_final_not_cleaned.to_excel('D:/training_result_notcleaned.xlsx')

Final model

In [17]:
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score
final_model = SpectralClustering(n_clusters=3, affinity='rbf', gamma=1.0, random_state=42)
final_labels_clean = final_model.fit_predict(X_clean)

# --- BƯỚC 2: TÍNH TOÁN CHỈ SỐ (Chỉ tính trên tập sạch) ---
s_score = silhouette_score(X_clean, final_labels_clean)
dbi_score = davies_bouldin_score(X_clean, final_labels_clean)

counts_clean = Counter(final_labels_clean)
cluster_sizes = {f"C{k}": v for k, v in sorted(counts_clean.items())}

user_data['final_label'] = -1
# --- BƯỚC 3: Mapping kết quả Spectral Clustering ---
# Chú ý: final_model.labels_ có độ dài bằng với số lượng True trong mask_clean
mask_clean = (user_data['dbscan_labels'] != -1)

# Sử dụng .values để đảm bảo gán theo thứ tự dòng, tránh lỗi lệch Index
user_data.loc[mask_clean, 'final_label'] = final_model.labels_

# --- BƯỚC 4: Kiểm tra kết quả gộp ---
print("Thống kê nhãn cuối cùng")
print(user_data['final_label'].value_counts())

Thống kê nhãn cuối cùng
final_label
 0    4107
 1    2533
 2    1167
-1     268
Name: count, dtype: int64


Save model

In [ ]:
# import os
# import joblib
#
# model_path = '../../resources/model/clustering/'
# if not os.path.exists(model_path):
#     os.makedirs(model_path)
#
# # Lưu Preprocessor
# joblib.dump(preprocessor, os.path.join(model_path, 'scaler_clustering_model.pkl'))
#
# # Lưu Model
# joblib.dump(final_model, os.path.join(model_path, 'spectral_clustering_model.pkl'))
#
# print("✅ Đã xuất Model và Scaler vào thư mục thành công!")

Prediction modeling

Preparing data

In [23]:
journey_data_compressed = journey_data_compressed.sort_values(
    by=['UserID', 'TIMESPSS'],
    ascending=[True, False]
)

# Tạo cột 'step' đánh số thứ tự các lần chạm của mỗi User
journey_data_compressed['step'] = journey_data_compressed.groupby('UserID').cumcount() + 1

In [24]:
journey_data_compressed

,UserID,type_touch,TIMESPSS,Duration,PurchaseID,DEVICE_TYPE,purchase_own,purchase_any,MobilePanel,FixedPanel,touchpoint_name,step
2,1,16,2016-05-21 08:39:44,103.679,5,FIXED,0,0,0,17,generic search,1
1,1,1,2015-11-03 12:31:35,922.000,4,FIXED,0,0,0,17,Accomodations Website,2
0,1,7,2015-08-07 19:16:59,120.000,3,FIXED,0,0,0,17,Touroperator / Travel agent Website Competitor,3
7,2,4,2015-09-02 08:41:00,84.000,6,MOBILE,0,0,17,0,Information / comparison Website,1
6,2,16,2015-08-15 22:57:31,111.000,6,MOBILE,0,0,17,0,generic search,2
...,...,...,...,...,...,...,...,...,...,...,...,...
303078,9676,1,2016-03-14 21:55:54,36.854,12538,FIXED,0,1,0,0,Accomodations Website,3
303083,9677,16,2016-07-26 10:08:17,64.471,25710,FIXED,0,0,0,0,generic search,1
303082,9677,1,2016-07-25 10:51:14,32.261,25710,FIXED,0,0,0,0,Accomodations Website,2
303081,9677,7,2016-06-20 09:07:22,483.003,25709,FIXED,0,0,0,0,Touroperator / Travel agent Website Competitor,3


In [25]:
journey_data_compressed = journey_data_compressed[journey_data_compressed['step'] <=3]
df_pivot = journey_data_compressed[['UserID', 'step', 'type_touch']]
# Index là UserID, Cột là các step (1, 2, 3), Giá trị là type_touch
df_step_pivot = journey_data_compressed.pivot(index='UserID', columns='step', values='type_touch')

In [26]:
df_step_pivot

step,1,2,3
UserID,,,
1,16.0,1.0,7.0
2,4.0,16.0,4.0
4,7.0,16.0,1.0
5,1.0,7.0,4.0
6,7.0,4.0,1.0
...,...,...,...
9674,1.0,7.0,1.0
9675,1.0,4.0,1.0
9676,1.0,16.0,1.0


In [27]:
# 2. Đổi tên cột từ 1, 2, 3 thành step1, step2, step3
df_step_pivot.columns = [f'step{int(c)}' for c in df_step_pivot.columns]

In [28]:
# 3. Reset index để đưa UserID trở lại thành một cột bình thường
df_final_steps = df_step_pivot.reset_index()
df_final_steps = df_final_steps[~df_final_steps['step2'].isnull() & ~df_final_steps['step3'].isnull()]
# Danh sách các cột thông tin người dùng cần lấy
selected_user_info = [
    'UserID',
    'GenderID',
    'Age',
    'SPSS_Regio5',
    'BAS_werkzaamheid_resp',
    'afg_kinderen_huishouden',
    'final_label'
]

# Mapping thông tin người dùng vào bảng hành vi
final_modeling_df = df_final_steps.merge(
    user_data[selected_user_info],
    on='UserID',
    how='inner'
)

In [29]:
final_modeling_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6183 entries, 0 to 6182
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UserID                   6183 non-null   int64  
 1   step1                    6183 non-null   float64
 2   step2                    6183 non-null   float64
 3   step3                    6183 non-null   float64
 4   GenderID                 6183 non-null   float64
 5   Age                      6183 non-null   float64
 6   SPSS_Regio5              6183 non-null   float64
 7   BAS_werkzaamheid_resp    6183 non-null   float64
 8   afg_kinderen_huishouden  6183 non-null   float64
 9   final_label              6183 non-null   int64  
dtypes: float64(8), int64(2)
memory usage: 483.2 KB


In [ ]:
import pandas as pd
import numpy as np
import warnings
from collections import Counter
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
import time

warnings.filterwarnings('ignore')

def smote(X, y, k_neighbors=5):
    X_res, y_res = X.copy(), list(y)
    counts = Counter(y)
    max_count = max(counts.values())
    for label, count in counts.items():
        if count == max_count or count < 2: continue
        X_class = X[y == label]
        actual_k = min(k_neighbors, count - 1)
        knn = NearestNeighbors(n_neighbors=actual_k).fit(X_class)
        num_to_add = max_count - count
        for _ in range(num_to_add):
            idx = np.random.randint(0, count)
            neighbor_idx = np.random.choice(knn.kneighbors([X_class[idx]], return_distance=False)[0])
            diff = X_class[neighbor_idx] - X_class[idx]
            X_res = np.vstack([X_res, X_class[idx] + np.random.rand() * diff])
            y_res.append(label)
    return X_res, np.array(y_res)

def tomek_links(X, y):
    knn = NearestNeighbors(n_neighbors=2).fit(X)
    nn_idx = knn.kneighbors(X, return_distance=False)
    to_delete = []
    for i in range(len(X)):
        neighbor = nn_idx[i, 1]
        if nn_idx[neighbor, 1] == i and y[i] != y[neighbor]:
            to_delete.append(i)
    mask = np.ones(len(y), dtype=bool)
    mask[to_delete] = False
    return X[mask], y[mask]

def adasyn(X, y, k_neighbors=5):
    X_res, y_res = X.copy(), list(y)
    counts = Counter(y)
    max_count = max(counts.values())

    # Lấy nhãn lớp đa số
    majority_label = max(counts, key=counts.get)
    X_majority = X[y == majority_label]

    # Tìm KNN trên toàn bộ dữ liệu để tính độ khó
    knn_full = NearestNeighbors(n_neighbors=k_neighbors).fit(X)

    for label, count in counts.items():
        if label == majority_label or count < 2: continue

        X_class = X[y == label]
        num_to_add_total = max_count - count

        # Bước 1: Tính tỉ lệ mẫu đa số xung quanh mỗi mẫu thiểu số (độ khó)
        indices = knn_full.kneighbors(X_class, return_distance=False)
        r = []
        for idx_list in indices:
            # Đếm xem trong k hàng xóm, có bao nhiêu thằng thuộc lớp đa số
            num_majority = sum(1 for i in idx_list if y[i] == majority_label)
            r.append(num_majority / k_neighbors)

        # Chuẩn hóa r để tổng bằng 1
        r = np.array(r)
        if np.sum(r) == 0: # Nếu lớp này quá tách biệt, quay về chia đều như SMOTE
            r = np.ones(len(r)) / len(r)
        else:
            r = r / np.sum(r)

        # Bước 2: Tính số mẫu cần tạo cho từng điểm dựa trên độ khó
        g = np.round(r * num_to_add_total).astype(int)

        # Bước 3: Tạo mẫu nội suy (tương tự SMOTE)
        knn_class = NearestNeighbors(n_neighbors=min(k_neighbors, count - 1)).fit(X_class)
        for i in range(len(X_class)):
            if g[i] == 0: continue

            # Tìm hàng xóm trong cùng lớp
            neighbors = knn_class.kneighbors([X_class[i]], return_distance=False)[0]

            for _ in range(g[i]):
                neighbor_idx = np.random.choice(neighbors)
                diff = X_class[neighbor_idx] - X_class[i]
                new_sample = X_class[i] + np.random.rand() * diff
                X_res = np.vstack([X_res, new_sample])
                y_res.append(label)

    return X_res, np.array(y_res)

def train_multi_model(data_name, X, y):
    """
    Hàm huấn luyện tất cả các mô hình trên từng biến thể dữ liệu.
    data_variants: Dictionary dạng {"Tên bộ dữ liệu": (X, y)}
    """

    models = {
        "Gradient-Boosted": GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
        "Deep Learning": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
        "Naïve Bayes": GaussianNB(),
        "SGD Classifier": SGDClassifier(loss='hinge', random_state=42)
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scoring_metrics = {
        'accuracy': 'accuracy',
        'precision': 'precision_weighted',
        'recall': 'recall_weighted',
        'f1': 'f1_weighted'
    }

    final_results = []

    print(f"{'='*60}")
    print(f"BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN")
    print(f"{'='*60}")

    print(f"\n>>> Thông tin bộ dữ liệu: {data_name} | Kích thước: {X.shape}")

    for model_name, model in models.items():
        try:
            print(model_name)
            start_t = time.perf_counter()
            # Tính toán cross-validation
            cv_results = cross_validate(
                model, X, y,
                cv=cv,
                scoring=scoring_metrics,
                error_score='raise'
            )
            end_t = time.perf_counter()
            total_time = end_t - start_t

            # Lưu kết quả
            r = {
                "Data Variant": data_name,
                "Model": model_name,
                "Accuracy": np.mean(cv_results['test_accuracy']),
                "Std Dev": np.std(cv_results['test_accuracy']),
                "Precision": np.mean(cv_results['test_precision']),
                "Recall": np.mean(cv_results['test_recall']),
                "F1-Score": np.mean(cv_results['test_f1']),
                "Time (s)": np.mean(cv_results['fit_time'])
            }

            final_results.append(r)
            print(f"  + {model_name.ljust(20)}: Hoàn tất (Accuracy: {r['Accuracy']:.4f} | F1: {r['F1-Score']:.4f} | Time : {total_time} )")

        except Exception as e:
            print(f"  + {model_name.ljust(20)}: LỖI - {str(e)[:50]}...")

    #XUẤT KẾT QUẢ
    df_final = pd.DataFrame(final_results)

    # Định dạng bảng
    df_display = df_final.copy()
    for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
        df_display[col] = df_display[col].apply(lambda x: f"{x*100:.2f}%")

    # Thêm cột Std Dev vào Accuracy
    df_display['Accuracy'] = df_display['Accuracy'] + " (±" + (df_final['Std Dev']*100).map("{:.2f}%".format) + ")"
    df_display = df_display.drop(columns=['Std Dev'])

    # Sắp xếp theo F1-Score hoặc Accuracy
    df_display = df_display.sort_values(by=['Data Variant', 'F1-Score'], ascending=[True, False])

    print("\n" + "="*80)
    print("BẢNG SO SÁNH HIỆU QUẢ TỔNG THỂ")
    print("="*80)
    print(df_display.to_string(index=False))
    return df_final

In [30]:

#CHUẨN BỊ DỮ LIỆU

# Nhóm dùng RobustScaler
robust_features = ['BAS_werkzaamheid_resp', 'afg_kinderen_huishouden']

# Nhóm dùng MinMaxScaler
minmax_features = ['GenderID', 'SPSS_Regio5', 'final_label', 'step2', 'step3']

# Nhóm dùng StandardScaler
standard_features = ['Age']

target = 'step1'

#XÂY DỰNG BỘ PREPROCESSOR
preprocessor = ColumnTransformer(
    transformers=[
        ('rob', RobustScaler(), robust_features),
        ('mms', MinMaxScaler(), minmax_features),
        ('std', StandardScaler(), standard_features)
    ],
    remainder='drop' # Tự động loại bỏ UserID và các cột thừa
)

#CHUẨN HÓA DỮ LIỆU
# X_scaled_raw sẽ là ma trận numpy chỉ chứa các features đã scale
X_scaled_raw = preprocessor.fit_transform(final_modeling_df)

# Lấy nhãn mục tiêu (Target)
y_raw = final_modeling_df[target].values

#LỌC LỚP HIẾM (Để SMOTE và CV không bị lỗi)
#Lọc dựa trên y_raw để đảm bảo mọi lớp đều có ít nhất 6 mẫu
min_samples = 6
counts = Counter(y_raw)
valid_classes = [cls for cls, count in counts.items() if count >= min_samples]

# Tạo mặt nạ (mask) để lọc đồng thời cả X và y
mask = np.isin(y_raw, valid_classes)

# ĐÂY LÀ 2 BIẾN CUỐI CÙNG CẬU CẦN ĐỂ NẠP VÀO MODEL
X_filtered = X_scaled_raw[mask]
y_filtered = y_raw[mask]

#KIỂM TRA KẾT QUẢ
print(f"Dữ liệu sẵn sàng")
print(f"--- Tổng số mẫu sau lọc: {len(y_filtered)}")
print(f"--- Số lượng features: {X_filtered.shape[1]}")
print(f"--- Danh sách lớp mục tiêu hợp lệ: {valid_classes}")

#TẠO CÁC BỘ DỮ LIỆU BIẾN THỂ
print("Đang tiền xử lý Resampling...")
X_sm, y_sm = smote(X_filtered, y_filtered)
X_tm, y_tm = tomek_links(X_filtered, y_filtered)
X_st, y_st = tomek_links(X_sm, y_sm)
X_ad, y_ad = adasyn(X_filtered, y_filtered)

data_variants = {
    "Original": (X_filtered, y_filtered),
    "SMOTE": (X_sm, y_sm),
    "Tomek-Links": (X_tm, y_tm),
    "SMOTE-Tomek": (X_st, y_st),
    "ADASYN": (X_ad, y_ad)
}

print('Đã chuẩn bị dữ liệu xong!')

Dữ liệu sẵn sàng
--- Tổng số mẫu sau lọc: 6180
--- Số lượng features: 8
--- Danh sách lớp mục tiêu hợp lệ: [16.0, 4.0, 7.0, 1.0, 2.0, 3.0, 5.0, 9.0, 10.0, 13.0, 14.0, 15.0, 6.0, 12.0]
Đang tiền xử lý Resampling...
Đã chuẩn bị dữ liệu xong!


In [34]:
#Original
X = data_variants['Original'][0]
y = data_variants['Original'][1]
df_original = train_multi_model('Original', X, y)

#SMOTE
X = data_variants['SMOTE'][0]
y = data_variants['SMOTE'][1]
df_SMOTE = train_multi_model('SMOTE', X, y)

#Tomek-Links
X = data_variants['Tomek-Links'][0]
y = data_variants['Tomek-Links'][1]
df_Tomek_Links = train_multi_model('Tomek-Links', X, y)

#SMOTE-Tomek
X = data_variants['SMOTE-Tomek'][0]
y = data_variants['SMOTE-Tomek'][1]
df_SMOTE_Tomek = train_multi_model('SMOTE-Tomek', X, y)

#ADASYN
X = data_variants['ADASYN'][0]
y = data_variants['ADASYN'][1]
df_ADASYN = train_multi_model('ADASYN', X, y)

df_final =  pd.concat([df_original, df_SMOTE, df_Tomek_Links, df_SMOTE_Tomek, df_ADASYN], ignore_index=True)

BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN

>>> Thông tin bộ dữ liệu: SMOTE-Tomek | Kích thước: (27870, 8)
Gradient-Boosted
  + Gradient-Boosted    : Hoàn tất (Accuracy: 0.7734 | F1: 0.7678 | Time : 296.3903292000032 )
Logistic Regression
  + Logistic Regression : Hoàn tất (Accuracy: 0.2372 | F1: 0.2018 | Time : 2.0604334000090603 )
Random Forest
  + Random Forest       : Hoàn tất (Accuracy: 0.7467 | F1: 0.7334 | Time : 13.204852700000629 )
Deep Learning
  + Deep Learning       : Hoàn tất (Accuracy: 0.6953 | F1: 0.6744 | Time : 254.8084927000018 )
Naïve Bayes
  + Naïve Bayes         : Hoàn tất (Accuracy: 0.2327 | F1: 0.2006 | Time : 0.17899450000550132 )
SGD Classifier
  + SGD Classifier      : Hoàn tất (Accuracy: 0.1279 | F1: 0.0932 | Time : 2.7164621999982046 )

BẢNG SO SÁNH HIỆU QUẢ TỔNG THỂ
Data Variant               Model        Accuracy Precision Recall F1-Score  Time (s)
 SMOTE-Tomek      SGD Classifier 12.79% (±0.87%)    11.33% 12.79%    9.32%  0.875295
 SMOTE-Tomek    Gradient-Boosted 77.3

In [38]:
df_final.to_excel('D:/MASTER/2.1. ADM/Test file/predict_training_result(4).xlsx')

Train for each cluster

In [40]:
import pandas as pd
import numpy as np
from collections import Counter

def split_data_by_clusters(df, cluster_col):
    """
    Tách DataFrame gốc thành dictionary các DataFrame con theo cluster.
    """
    unique_clusters = sorted(df[cluster_col].unique())
    cluster_dict = {}

    print(f"{'='*50}")
    print(f"BƯỚC 1: CHIA DỮ LIỆU THEO CLUSTER")
    for cid in unique_clusters:
        name = f"cluster_{cid}" if cid != -1 else "cluster_noise"
        cluster_dict[name] = df[df[cluster_col] == cid].copy()
        print(f"  + {name.ljust(15)}: {len(cluster_dict[name])} dòng")
    return cluster_dict

# --- HÀM XỬ LÝ SCALING VÀ RESAMPLING CHO TỪNG CLUSTER ---
def preprocess_clusters(cluster_dict, target_col, preprocessor):
    """
    Lặp qua từng cluster để:
    1. Scale dữ liệu
    2. Lọc lớp hiếm
    3. Tạo các biến thể Resampling (SMOTE, ADASYN,...)
    """
    processed_results = {}

    print(f"\n{'='*50}")
    print(f"BƯỚC 2: SCALING & RESAMPLING CHI TIẾT")

    for cluster_name, df_raw in cluster_dict.items():
        if len(df_raw) < 10:
            print(f"  ! {cluster_name}: Bỏ qua (quá ít dữ liệu)")
            continue

        print(f"  > Đang xử lý {cluster_name}...")

        # 1. Scaling
        # Lưu ý: fit_transform tại đây giúp scaler học đặc trưng riêng của từng cluster
        X_scaled = preprocessor.fit_transform(df_raw)
        y = df_raw[target_col].values

        # 2. Lọc lớp hiếm (để tránh lỗi khi chạy SMOTE/ADASYN)
        min_samples = 6
        counts = Counter(y)
        valid_classes = [cls for cls, count in counts.items() if count >= min_samples]

        if len(valid_classes) < 2:
            print(f"    - {cluster_name}: Không đủ lớp mục tiêu để Resampling.")
            continue

        mask = np.isin(y, valid_classes)
        X_f = X_scaled[mask]
        y_f = y[mask]

        # 3. Tạo các biến thể Resampling (sử dụng các hàm manual bạn đã viết)
        try:
            X_sm, y_sm = smote(X_f, y_f)
            X_tm, y_tm = tomek_links(X_f, y_f)
            X_st, y_st = tomek_links(X_sm, y_sm)
            X_ad, y_ad = adasyn(X_f, y_f)

            # Lưu vào dictionary kết quả cuối cùng
            processed_results[cluster_name] = {
                "Original": (X_f, y_f),
                "SMOTE": (X_sm, y_sm),
                "Tomek": (X_tm, y_tm),
                "SMOTE-Tomek": (X_st, y_st),
                "ADASYN": (X_ad, y_ad)
            }
            print(f"    - {cluster_name}: Hoàn tất tạo 5 biến thể dữ liệu.")

        except Exception as e:
            print(f"    - {cluster_name}: Lỗi Resampling - {str(e)}")

    return processed_results

# --- THỰC THI TỔNG THỂ ---

# 1. Chia cluster thô
cluster_raw_dict = split_data_by_clusters(final_modeling_df, 'final_label')

# 2. Xử lý chuyên sâu cho từng cluster

# Nhóm dùng RobustScaler
robust_features = ['BAS_werkzaamheid_resp', 'afg_kinderen_huishouden']

# Nhóm dùng MinMaxScaler
minmax_features = ['GenderID', 'SPSS_Regio5', 'step2', 'step3']

# Nhóm dùng StandardScaler
standard_features = ['Age']

target = 'step1'

#XÂY DỰNG BỘ PREPROCESSOR
preprocessor_cluster = ColumnTransformer(
    transformers=[
        ('rob', RobustScaler(), robust_features),
        ('mms', MinMaxScaler(), minmax_features),
        ('std', StandardScaler(), standard_features)
    ],
    remainder='drop' # Tự động loại bỏ UserID và các cột thừa
)

final_processed_data = preprocess_clusters(
    cluster_dict=cluster_raw_dict,
    target_col='step1',
    preprocessor=preprocessor_cluster
) # Output là một dictionary lồng nhau: final_processed_data['cluster_0']['SMOTE'] -> (X, y)

print(f"\n{'='*50}")
print("TẤT CẢ DỮ LIỆU ĐÃ SẴN SÀNG ĐỂ TRAINING!")

BƯỚC 1: CHIA DỮ LIỆU THEO CLUSTER
  + cluster_noise  : 190 dòng
  + cluster_0      : 3170 dòng
  + cluster_1      : 1900 dòng
  + cluster_2      : 923 dòng

BƯỚC 2: SCALING & RESAMPLING CHI TIẾT
  > Đang xử lý cluster_noise...
    - cluster_noise: Hoàn tất tạo 5 biến thể dữ liệu.
  > Đang xử lý cluster_0...
    - cluster_0: Hoàn tất tạo 5 biến thể dữ liệu.
  > Đang xử lý cluster_1...
    - cluster_1: Hoàn tất tạo 5 biến thể dữ liệu.
  > Đang xử lý cluster_2...
    - cluster_2: Hoàn tất tạo 5 biến thể dữ liệu.

TẤT CẢ DỮ LIỆU ĐÃ SẴN SÀNG ĐỂ TRAINING!


Cluster 0

In [44]:
X, y = final_processed_data['cluster_0']['Original']
df_Original_0 = train_multi_model('cluster_0_Original', X, y)
X, y = final_processed_data['cluster_0']['SMOTE']
df_SMOTE_0 = train_multi_model('cluster_0_SMOTE', X, y)
X, y = final_processed_data['cluster_0']['Tomek']
df_Tomek_0 = train_multi_model('cluster_0_Tomek', X, y)
X, y = final_processed_data['cluster_0']['ADASYN']
df_ADASYN_0 = train_multi_model('cluster_0_ADASYN', X, y)
X, y = final_processed_data['cluster_0']['SMOTE-Tomek']
df_SMOTE_Tomek_0 = train_multi_model('cluster_0_SMOTE_Tomek', X, y)
df_final_0= pd.concat([df_Original_0, df_SMOTE_0, df_Tomek_0, df_ADASYN_0, df_SMOTE_Tomek_0], ignore_index=True)
df_final_0.to_excel('D:/MASTER/2.1. ADM/Test file/predict_training_result_c0(5).xlsx')

BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN

>>> Thông tin bộ dữ liệu: cluster_0_Original | Kích thước: (3160, 7)
Gradient-Boosted
  + Gradient-Boosted    : Hoàn tất (Accuracy: 0.5095 | F1: 0.4800 | Time : 23.654015699998126 )
Logistic Regression
  + Logistic Regression : Hoàn tất (Accuracy: 0.5051 | F1: 0.4007 | Time : 0.3327409999910742 )
Random Forest
  + Random Forest       : Hoàn tất (Accuracy: 0.5443 | F1: 0.4887 | Time : 1.6475663000019267 )
Deep Learning
  + Deep Learning       : Hoàn tất (Accuracy: 0.5329 | F1: 0.4900 | Time : 31.673623199996655 )
Naïve Bayes
  + Naïve Bayes         : Hoàn tất (Accuracy: 0.0626 | F1: 0.0261 | Time : 0.06628030000138097 )
SGD Classifier
  + SGD Classifier      : Hoàn tất (Accuracy: 0.4636 | F1: 0.3736 | Time : 0.5320422999939183 )

BẢNG SO SÁNH HIỆU QUẢ TỔNG THỂ
      Data Variant               Model        Accuracy Precision Recall F1-Score  Time (s)
cluster_0_Original       Deep Learning 53.29% (±1.18%)    47.80% 53.29%   49.00% 10.540060
cluster_0_Origina

Cluster 1

In [51]:
X, y = final_processed_data['cluster_1']['Original']
df_Original_1 = train_multi_model('cluster_0_Original', X, y)

X, y = final_processed_data['cluster_1']['SMOTE']
df_SMOTE_1 = train_multi_model('cluster_1_SMOTE', X, y)

X, y = final_processed_data['cluster_1']['Tomek']
df_Tomek_1 = train_multi_model('cluster_1_Tomek', X, y)

X, y = final_processed_data['cluster_1']['ADASYN']
df_ADASYN_1 = train_multi_model('cluster_1_ADASYN', X, y)

X, y = final_processed_data['cluster_1']['SMOTE-Tomek']
df_SMOTE_Tomek_1 = train_multi_model('cluster_1_SMOTE_Tomek', X, y)

df_final_1= pd.concat([df_Original_1, df_SMOTE_1, df_Tomek_1, df_ADASYN_1, df_SMOTE_Tomek_1], ignore_index=True)

df_final_1.to_excel('D:/MASTER/2.1. ADM/Test file/predict_training_result_c1(5).xlsx')

BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN

>>> Thông tin bộ dữ liệu: cluster_0_Original | Kích thước: (1894, 7)
Gradient-Boosted
  + Gradient-Boosted    : Hoàn tất (Accuracy: 0.4805 | F1: 0.4579 | Time : 21.42402160000347 )
Logistic Regression
  + Logistic Regression : Hoàn tất (Accuracy: 0.4863 | F1: 0.3921 | Time : 0.3122340000118129 )
Random Forest
  + Random Forest       : Hoàn tất (Accuracy: 0.5333 | F1: 0.4699 | Time : 1.492627800005721 )
Deep Learning
  + Deep Learning       : Hoàn tất (Accuracy: 0.4662 | F1: 0.4243 | Time : 17.63033749999886 )
Naïve Bayes
  + Naïve Bayes         : Hoàn tất (Accuracy: 0.4979 | F1: 0.4216 | Time : 0.06875170000421349 )
SGD Classifier
  + SGD Classifier      : Hoàn tất (Accuracy: 0.3960 | F1: 0.3076 | Time : 0.4122400999913225 )

BẢNG SO SÁNH HIỆU QUẢ TỔNG THỂ
      Data Variant               Model        Accuracy Precision Recall F1-Score  Time (s)
cluster_0_Original       Random Forest 53.33% (±0.36%)    45.93% 53.33%   46.99%  0.442362
cluster_0_Original  

Cluster 2

In [52]:
X, y = final_processed_data['cluster_2']['Original']
df_Original_2 = train_multi_model('cluster_2_Original', X, y)

X, y = final_processed_data['cluster_2']['SMOTE']
df_SMOTE_2 = train_multi_model('cluster_2_SMOTE', X, y)

X, y = final_processed_data['cluster_2']['Tomek']
df_Tomek_2 = train_multi_model('cluster_2_Tomek', X, y)

X, y = final_processed_data['cluster_2']['ADASYN']
df_ADASYN_2 = train_multi_model('cluster_2_ADASYN', X, y)

X, y = final_processed_data['cluster_2']['SMOTE-Tomek']
df_SMOTE_Tomek_2 = train_multi_model('cluster_2_SMOTE_Tomek', X, y)

df_final_2= pd.concat([df_Original_2, df_SMOTE_2, df_Tomek_2, df_ADASYN_2, df_SMOTE_Tomek_2], ignore_index=True)

df_final_2.to_excel('D:/MASTER/2.1. ADM/Test file/predict_training_result_c2(5).xlsx')

BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN

>>> Thông tin bộ dữ liệu: cluster_2_Original | Kích thước: (910, 7)
Gradient-Boosted
  + Gradient-Boosted    : Hoàn tất (Accuracy: 0.3571 | F1: 0.3413 | Time : 17.971547799999826 )
Logistic Regression
  + Logistic Regression : Hoàn tất (Accuracy: 0.3715 | F1: 0.3015 | Time : 0.19516040000598878 )
Random Forest
  + Random Forest       : Hoàn tất (Accuracy: 0.3846 | F1: 0.3473 | Time : 1.3780021999991732 )
Deep Learning
  + Deep Learning       : Hoàn tất (Accuracy: 0.3912 | F1: 0.3259 | Time : 7.591731799999252 )
Naïve Bayes
  + Naïve Bayes         : Hoàn tất (Accuracy: 0.2933 | F1: 0.2753 | Time : 0.07243479999306146 )
SGD Classifier
  + SGD Classifier      : Hoàn tất (Accuracy: 0.2385 | F1: 0.2074 | Time : 0.19572580000385642 )

BẢNG SO SÁNH HIỆU QUẢ TỔNG THỂ
      Data Variant               Model         Accuracy Precision Recall F1-Score  Time (s)
cluster_2_Original       Random Forest  38.46% (±1.63%)    32.89% 38.46%   34.73%  0.407759
cluster_2_Origi

CLuster noise

In [53]:
X, y = final_processed_data['cluster_noise']['Original']
df_Original_noise = train_multi_model('cluster_noise_Original', X, y)

X, y = final_processed_data['cluster_noise']['SMOTE']
df_SMOTE_noise = train_multi_model('cluster_noise_SMOTE', X, y)

X, y = final_processed_data['cluster_noise']['Tomek']
df_Tomek_noise = train_multi_model('cluster_noise_Tomek', X, y)

X, y = final_processed_data['cluster_noise']['ADASYN']
df_ADASYN_noise = train_multi_model('cluster_noise_ADASYN', X, y)

X, y = final_processed_data['cluster_noise']['SMOTE-Tomek']
df_SMOTE_Tomek_noise = train_multi_model('cluster_noise_SMOTE_Tomek', X, y)

df_final_2= pd.concat([df_Original_noise, df_SMOTE_noise, df_Tomek_noise, df_ADASYN_noise, df_SMOTE_Tomek_noise], ignore_index=True)

df_final_2.to_excel('D:/MASTER/2.1. ADM/Test file/predict_training_result_cnoise(5).xlsx')

BẮT ĐẦU TIẾN TRÌNH HUẤN LUYỆN

>>> Thông tin bộ dữ liệu: cluster_noise_Original | Kích thước: (184, 7)
Gradient-Boosted
  + Gradient-Boosted    : Hoàn tất (Accuracy: 0.3858 | F1: 0.3660 | Time : 6.547014700001455 )
Logistic Regression
  + Logistic Regression : Hoàn tất (Accuracy: 0.3860 | F1: 0.3273 | Time : 0.08941960000083782 )
Random Forest
  + Random Forest       : Hoàn tất (Accuracy: 0.4292 | F1: 0.3957 | Time : 1.044553999992786 )
Deep Learning
  + Deep Learning       : Hoàn tất (Accuracy: 0.3532 | F1: 0.3343 | Time : 2.186658599996008 )
Naïve Bayes
  + Naïve Bayes         : Hoàn tất (Accuracy: 0.3482 | F1: 0.3387 | Time : 0.05262539999966975 )
SGD Classifier
  + SGD Classifier      : Hoàn tất (Accuracy: 0.3423 | F1: 0.3066 | Time : 0.07100130000617355 )

BẢNG SO SÁNH HIỆU QUẢ TỔNG THỂ
          Data Variant               Model        Accuracy Precision Recall F1-Score  Time (s)
cluster_noise_Original       Random Forest 42.92% (±2.56%)    41.09% 42.92%   39.57%  0.307386
cluster

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier

# --- 1. ĐƯỜNG DẪN MÔ HÌNH ---
model_path = '../../resources/model/clustering/'
dbscan_file = os.path.join(model_path, 'dbscan_clustering_model.pkl')
preprocessor_file = os.path.join(model_path, 'user_data_preprocessor.pkl')
gbc_file = os.path.join(model_path, 'gradient_boosting_cluster_classifier.pkl')

# --- 2. HUẤN LUYỆN GRADIENT BOOSTING CŨNG VỚI CÁC CỤM DBSCAN ---
def train_gbc_classifier(X_train_scaled):
    """
    Sử dụng Gradient Boosting để học lại phân bổ cụm từ DBSCAN
    (Chỉ cần chạy hàm này 1 lần duy nhất sau khi bạn train xong DBSCAN)
    """
    # Tải model DBSCAN hiện tại lên để lấy nhãn
    dbscan = joblib.load(dbscan_file)
    labels = dbscan.labels_

    # Loại bỏ các điểm nhiễu (-1) vì Gradient Boosting chỉ cần học các cụm hợp lệ
    valid_mask = labels != -1
    X_valid = X_train_scaled[valid_mask]
    y_valid = labels[valid_mask]

    # Khởi tạo và huấn luyện Gradient Boosting Classifier
    print("⏳ Đang huấn luyện Gradient Boosting Classifier trên các cụm DBSCAN...")
    gbc = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
    gbc.fit(X_valid, y_valid)

    # Lưu lại model GBC
    joblib.dump(gbc, gbc_file)
    print("✅ Đã huấn luyện và lưu Gradient Boosting Classifier thành công!")


# --- 3. GỌI MODEL VÀ DỰ BÁO TOP 3 CÓ % CAO NHẤT ---
def predict_top_3_with_percentage(raw_new_data):
    """
    Gọi các mô hình lên, tiền xử lý dữ liệu mới và xuất ra top 3 cụm có tỷ lệ % cao nhất.
    """
    # Tải lại Bộ tiền xử lý và Bộ phân lớp Gradient Boosting
    preprocessor = joblib.load(preprocessor_file)
    gbc = joblib.load(gbc_file)

    # Step 1: Chuẩn hóa dữ liệu mới (Scale dữ liệu về dạng khớp với lúc train)
    scaled_new_data = preprocessor.transform(raw_new_data)

    # Step 2: Tính toán xác suất (%) lọt vào các cụm bằng Gradient Boosting
    # [0] nghĩa là lấy kết quả cho hàng dữ liệu đầu tiên được truyền vào
    probabilities = gbc.predict_proba(scaled_new_data)[0]
    cluster_classes = gbc.classes_

    # Step 3: Ghép cặp (Tên Cụm, % Xác suất) và sắp xếp giảm dần để lấy Top 3
    cluster_prob_pairs = list(zip(cluster_classes, probabilities))
    top_3 = sorted(cluster_prob_pairs, key=lambda x: x[1], reverse=True)[:3]

    # Hiển thị kết quả ra màn hình cực kỳ trực quan
    print("\n📊 === KẾT QUẢ DỰ BÁO GRADIENT BOOSTING (TOP 3) ===")
    for idx, (cluster, prob) in enumerate(top_3, 1):
        # Nhân với 100 để ra tỷ lệ phần trăm %
        percentage = prob * 100
        print(f"Top {idx} -> Cụm [ {cluster} ] với tỷ lệ chính xác: {percentage:.2f}%")

    return top_3

# --- 💡 CÁCH BẠN ÁP DỤNG TRONG MAIN.PY ---
# BƯỚC A: Chạy hàm này một lần ở file huấn luyện để tạo file .pkl cho GBC:
# train_gbc_classifier(X_train_scaled)

# BƯỚC B: Ở bất kỳ đâu bạn muốn dự báo (ví dụ nhận data từ người dùng nhập vào):
# X_new = pd.DataFrame([{ 'Age': 30, 'GenderID': 1, 'BAS_huishoudgrootte': 2, ... }]) # Data mẫu 1 dòng
# predict_top_3_with_percentage(X_new)